In [5]:
import sys
from pathlib import Path

# Add project root to Python path so we can import src modules
project_root = Path.cwd().parent   
sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
from src.data_loader import load_fear_greed_data, load_trader_data, merge_data, save_processed_data
from src.feature_engineering import compute_daily_metrics, add_trader_segments

print("Setup complete")

Setup complete


In [6]:
# Load raw data
fear = load_fear_greed_data()
trader = load_trader_data()

print("Fear & Greed data shape:", fear.shape)
display(fear.head())

print("\nTrader data shape:", trader.shape)
display(trader.head())

Fear & Greed data shape: (2644, 3)


,date,value,classification
0,2018-02-01,30,Fear
1,2018-02-02,15,Extreme Fear
2,2018-02-03,40,Fear
3,2018-02-04,24,Extreme Fear
4,2018-02-05,11,Extreme Fear



Trader data shape: (211224, 18)


,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,Transaction Hash,Order ID,Crossed,Fee,Trade ID,Timestamp,timestamp,date
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.87,7872.16,BUY,02-12-2024 22:50,0.000000,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.345404,8.950000e+14,1.730000e+12,2024-10-27 03:33:20,2024-10-27
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.00,127.68,BUY,02-12-2024 22:50,986.524596,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.005600,4.430000e+14,1.730000e+12,2024-10-27 03:33:20,2024-10-27
2,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9855,144.09,1150.63,BUY,02-12-2024 22:50,1002.518996,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050431,6.600000e+14,1.730000e+12,2024-10-27 03:33:20,2024-10-27
3,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9874,142.98,1142.04,BUY,02-12-2024 22:50,1146.558564,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050043,1.080000e+15,1.730000e+12,2024-10-27 03:33:20,2024-10-27
4,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9894,8.73,69.75,BUY,02-12-2024 22:50,1289.488521,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.003055,1.050000e+15,1.730000e+12,2024-10-27 03:33:20,2024-10-27


In [7]:
# Merge datasets
merged = merge_data(trader, fear)
print("Merged shape:", merged.shape)
print("Missing sentiment values after merge:", merged['classification'].isna().sum())
display(merged.head())

Merged shape: (211224, 20)
Missing sentiment values after merge: 0


C:\Users\naval\OneDrive\Desktop\hyperliquid_sentiment_analysis\src\data_loader.py:23: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  merged['classification'] = merged['classification'].fillna(method='ffill')


,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,Transaction Hash,Order ID,Crossed,Fee,Trade ID,Timestamp,timestamp,date,value,classification
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.87,7872.16,BUY,02-12-2024 22:50,0.000000,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.345404,8.950000e+14,1.730000e+12,2024-10-27 03:33:20,2024-10-27,74.0,Greed
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.00,127.68,BUY,02-12-2024 22:50,986.524596,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.005600,4.430000e+14,1.730000e+12,2024-10-27 03:33:20,2024-10-27,74.0,Greed
2,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9855,144.09,1150.63,BUY,02-12-2024 22:50,1002.518996,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050431,6.600000e+14,1.730000e+12,2024-10-27 03:33:20,2024-10-27,74.0,Greed
3,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9874,142.98,1142.04,BUY,02-12-2024 22:50,1146.558564,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050043,1.080000e+15,1.730000e+12,2024-10-27 03:33:20,2024-10-27,74.0,Greed
4,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9894,8.73,69.75,BUY,02-12-2024 22:50,1289.488521,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.003055,1.050000e+15,1.730000e+12,2024-10-27 03:33:20,2024-10-27,74.0,Greed


In [8]:
# Save processed merged data
save_processed_data(merged, "merged_trader_sentiment.parquet")
print("Saved to data/processed/merged_trader_sentiment.parquet")

Saved to data/processed/merged_trader_sentiment.parquet


In [9]:
# Compute daily metrics and trader segments
daily = compute_daily_metrics(merged)
daily = add_trader_segments(daily)
daily.to_parquet(project_root / "data" / "processed" / "daily_metrics.parquet", index=False)
print("Daily metrics shape:", daily.shape)
display(daily.head())

Daily metrics shape: (102, 13)


C:\Users\naval\OneDrive\Desktop\hyperliquid_sentiment_analysis\src\feature_engineering.py:27: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  drawdown = df.groupby(['date', 'Account']).apply(compute_daily_drawdown).reset_index(name='max_daily_drawdown')


,date,Account,daily_pnl,win_rate,trade_count,avg_trade_size_usd,avg_leverage,long_short_ratio,max_daily_drawdown,classification,leverage_segment,frequency_segment,consistency_segment
0,2023-03-28,0x3998f134d6aaa2b6a5f723806d00fd2bbbbce891,0.000000,0.000000,3,159.000000,1.0,3.000000,0.000000,Greed,Low (0-5x),Low freq,Low win rate (<40%)
1,2023-11-14,0x3998f134d6aaa2b6a5f723806d00fd2bbbbce891,0.000000,0.000000,2,23066.935000,1.0,2.000000,0.000000,Greed,Low (0-5x),Low freq,Low win rate (<40%)
2,2023-11-14,0xb1231a4a2dd02f2276fa3c5e2a2f3436e6bfed23,155.503357,0.275168,1043,11034.799511,1.0,0.882671,-892.507908,Greed,Low (0-5x),High freq,Low win rate (<40%)
3,2024-03-09,0x3998f134d6aaa2b6a5f723806d00fd2bbbbce891,-5564.016140,0.333333,27,3048.594444,1.0,0.173913,-5564.016140,Extreme Greed,Low (0-5x),Low freq,Low win rate (<40%)
4,2024-03-09,0x430f09841d65beb3f27765503d0f850b8bce7713,0.000000,0.000000,88,1136.312727,1.0,88.000000,0.000000,Extreme Greed,Low (0-5x),Low freq,Medium win rate (40-60%)


In [10]:
# Basic summary statistics
print("Daily metrics summary (descriptive stats):")
display(daily.describe())

print("Unique traders:", daily['Account'].nunique())
print("Date range:", daily['date'].min(), "to", daily['date'].max())

Daily metrics summary (descriptive stats):


,daily_pnl,win_rate,trade_count,avg_trade_size_usd,avg_leverage,long_short_ratio,max_daily_drawdown
count,1.020000e+02,102.000000,102.000000,102.000000,102.0,102.000000,102.000000
mean,1.009506e+05,0.386000,2070.823529,5771.203685,1.0,4.929462,-13579.937175
std,2.826451e+05,0.248853,3796.503132,7819.744671,0.0,24.216173,56691.188331
min,-3.639249e+05,0.000000,2.000000,7.020000,1.0,0.000000,-437570.038092
25%,1.265832e+02,0.244818,157.750000,1368.424745,1.0,0.658556,-789.733004
50%,1.848526e+04,0.387624,548.500000,2709.532198,1.0,0.891638,0.000000
75%,9.994411e+04,0.495262,3052.250000,6923.092523,1.0,1.416539,0.000000
max,1.927736e+06,1.000000,26651.000000,41153.695500,1.0,223.000000,1584.342755


Unique traders: 32
Date range: 2023-03-28 to 2025-06-15


In [11]:
# Segment distributions
print("\nLeverage segments:")
print(daily['leverage_segment'].value_counts())
print("\nFrequency segments:")
print(daily['frequency_segment'].value_counts())
print("\nConsistency segments:")
print(daily['consistency_segment'].value_counts())


Leverage segments:
leverage_segment
Low (0-5x)        102
Medium (5-10x)      0
High (10x+)         0
Name: count, dtype: int64

Frequency segments:
frequency_segment
Low freq       36
High freq      36
Medium freq    30
Name: count, dtype: int64

Consistency segments:
consistency_segment
Low win rate (<40%)         59
Medium win rate (40-60%)    34
High win rate (>60%)         9
Name: count, dtype: int64
